# Lab 2: Vectorless RAG — Advanced Scenarios


## Setup

In [ ]:
!pip install -q pageindex openai python-dotenv pymupdf

In [ ]:
import os
import json
import re
import fitz
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv("../.env")

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

print("Keys loaded.")

In [ ]:
def call_llm(prompt, model="meta-llama/llama-4-scout-17b-16e-instruct"):
    """Call a language model via OpenRouter."""
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
    )
    msgs = [{"role": "user", "content": prompt}]
    resp = client.chat.completions.create(model=model, messages=msgs, temperature=0, max_tokens=1024)
    return resp.choices[0].message.content.strip()

In [ ]:
# Point this to any PDF you want to query
PDF_PATH = "data/your_document.pdf"

doc = fitz.open(PDF_PATH)
page_texts = {i+1: doc.load_page(i).get_text() for i in range(len(doc))}
doc.close()
print(f"Extracted text from {len(page_texts)} pages.")

In [ ]:
from pageindex import PageIndexClient
from pageindex import utils
import time

CACHE_PATH = f"cache/{os.path.basename(PDF_PATH).replace('.pdf', '_tree.json')}"
os.makedirs("cache", exist_ok=True)

tree = None
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH) as f:
        tree = json.load(f)
    print("Loaded tree from cache.")

if tree is None:
    pi = PageIndexClient(api_key=PAGEINDEX_API_KEY)
    result = pi.submit_document(PDF_PATH)
    doc_id = result["doc_id"]
    print(f"Submitted: {doc_id}")

    elapsed = 0
    while elapsed < 300:
        if pi.is_retrieval_ready(doc_id):
            break
        time.sleep(5)
            elapsed += 5
        print(f"  {elapsed}s...")
    else:
        raise TimeoutError("PageIndex timeout")

    tree = pi.get_tree(doc_id, node_summary=True)["result"]
    with open(CACHE_PATH, "w") as f:
        json.dump(tree, f, indent=2)
    print("Tree cached.")

utils.print_tree(tree, exclude_fields=["text"])

---
## Helper Functions

In [ ]:
def parse_json(text):
    """Extract JSON from LLM response."""
    text = re.sub(r"```json\s*|\s*```", "", text.strip())
    s, e = text.find("{"), text.rfind("}")
    if s != -1 and e != -1:
        text = text[s:e+1]
    return json.loads(text)

def retrieve_nodes(query):
    """Use LLM to find relevant nodes for a query."""
    tree_slim = utils.remove_fields(tree.copy(), fields=["text"])
    prompt = f"""
Given a question and a document tree (node_id, title, summary),
find nodes likely to contain the answer.

Question: {query}

Tree:
{json.dumps(tree_slim, indent=2)}

JSON only:
{{"thinking": "...", "node_list": ["id1", "id2"]}}
"""
    return parse_json(call_llm(prompt))

def get_context(node_list):
    """Extract text from pages covered by the given nodes."""
    node_map = utils.create_node_mapping(tree, include_page_ranges=True, max_page=len(page_texts))
    texts, seen = [], set()
    for nid in node_list:
        info = node_map[nid]
        for p in range(info["start_index"], info["end_index"] + 1):
            if p not in seen and p in page_texts:
                texts.append(f"--- Page {p} ---\n{page_texts[p]}")
                seen.add(p)
    return "\n\n".join(texts)

print("Helpers ready.")

---
# Scenario 1: Multi-Hop Attribute Aggregation

**Problem:** Some questions require combining information from multiple sections. For example:
- "What is the total cost including tax?" (requires item price + tax rate)
- "What are the requirements and deadlines?" (requires specs + schedule)

**Solution:** The LLM must retrieve nodes from multiple sections and aggregate the information.

In [ ]:
# Change this to a question that requires info from multiple sections
QUERY_MULTI_HOP = "What is the total cost including all fees and taxes?"

In [ ]:
result = retrieve_nodes(QUERY_MULTI_HOP)
print("Reasoning:", result["thinking"], "\n")
print("Retrieved nodes:", result["node_list"])

In [ ]:
context = get_context(result["node_list"])
answer = call_llm(f"Context:\n{context}\n\nQuestion: {QUERY_MULTI_HOP}\n\nAnswer concisely.")
print("Answer:", answer)

### Why Multi-Hop is Hard

Traditional RAG retrieves chunks by similarity. A query about "total cost" might only match the pricing section OR the tax section — not both. Vectorless RAG with tree-based reasoning can identify that the answer requires **multiple sections**.

---
# Scenario 2: Structured Data Fidelity

**Problem:** Documents contain tables, forms, and structured data. Extracting this data accurately is critical — a wrong number can be costly.

**Solution:** The LLM reads the raw text (which preserves table structure) and extracts values with high fidelity.

In [ ]:
# Change this to a question about extracting specific values from a table
QUERY_STRUCTURED = "What are the limits or caps mentioned in the table?"

In [ ]:
result = retrieve_nodes(QUERY_STRUCTURED)
print("Reasoning:", result["thinking"], "\n")
print("Retrieved nodes:", result["node_list"])

In [ ]:
context = get_context(result["node_list"])
answer = call_llm(f"Context:\n{context}\n\nQuestion: {QUERY_STRUCTURED}\n\nAnswer concisely.")
print("Answer:", answer)

### Why Structured Data Fidelity Matters

Tables in PDFs are often poorly formatted when extracted. Vectorless RAG preserves the original text flow, allowing the LLM to understand table structure from context (column headers, row labels, etc.).

---
# Scenario 3: Combined — Multi-Hop + Structured Data

The hardest case: a question that requires **both** multi-hop reasoning AND accurate structured data extraction.

In [ ]:
# Change this to a question that needs both multi-hop and table extraction
QUERY_COMBINED = "What is the final amount after applying all adjustments from the tables?"

In [ ]:
result = retrieve_nodes(QUERY_COMBINED)
print("Reasoning:", result["thinking"], "\n")
print("Retrieved nodes:", result["node_list"])

In [ ]:
context = get_context(result["node_list"])
answer = call_llm(f"Context:\n{context}\n\nQuestion: {QUERY_COMBINED}\n\nAnswer concisely.")
print("Answer:", answer)

---
## Try It Yourself

Change the `QUERY_*` variables and re-run the cells. Here are some generic patterns to try:

In [ ]:
examples = [
    # Multi-hop
    "What are the requirements and how do they change over time?",
    "What are the sub-components and their respective limits?",
    
    # Structured data
    "List all items and their corresponding values from the table.",
    "What is the breakdown of costs by category?",
    
    # Combined
    "Calculate the total based on all the tables and sections.",
]

for q in examples:
    result = retrieve_nodes(q)
    print(f"{q[:50]}... -> {len(result['node_list'])} nodes")